# FSG 월간보고 엔진 검산 — PC 없이 (Colab)

Google 시트 「FSG_월_분기 보고 데이터」를 그대로 읽어 새 엔진(fsg-report)과 정제 탭 13개를 칸마다 비교합니다(설치 3탭은 2026-10-26부터 규칙이 달라 비교하지 않습니다).
데이터는 이 Colab(내 Google 계정 안)에서만 처리되고, 결과 요약(이름 없음)만 Drive 폴더 `FSG_report_out`에 저장됩니다.

**순서**: 메뉴 **런타임 › 모두 실행** → Google 계정 권한 허용 창에서 허용 → 3~5분 뒤 맨 아래 결과 확인.
(결과 파일은 Drive의 `FSG_report_out/검산결과_날짜.txt`에도 남으므로 Claude가 Drive 커넥터로 바로 읽을 수 있습니다.)

In [ ]:
#@title 1) 설정 — 보통 그대로 둡니다
SHEET_ID = '1OH2Zy1Hg-9JrfgpxfeRn4Xxa9xUkZkAxS89BRBkFvNs'   #@param {type:"string"}
BRANCH   = 'main'                         #@param {type:"string"}
PERIOD   = ''   #@param {type:"string"}  (비우면 시트 설정 B2 기간. 예: 2026-09, FY2026Q2)
CUT      = ''   #@param {type:"string"}  (비우면 시트 설정 B4 추출일. 예: 2026-10-12)
REPO_URL = 'https://github.com/balhm12/Sysmex-schedule.git'
TABS = ['설정','로스터','원본_활동','원본_Serial','실행로그',
        '정제_KPI','정제_KPI월별추이','정제_팀별실적','정제_케이스분석','정제_트렌드비교','정제_수도권',
        '정제_엔지니어생산성','정제_병원TOP5','정제_반복고장TOP5','정제_장애요인TOP4','정제_장비군에러율','정제_운영건전성','정제_집중관리대상']
XLSX, REPO, RESULT = '/content/sheet_copy.xlsx', '/content/sysmex-schedule', '/content/검산결과.txt'
print('설정 완료')

In [ ]:
#@title 2) Google 시트 읽어 엑셀 사본 만들기 (권한 창이 뜨면 허용)
import time
from google.colab import auth
auth.authenticate_user()
from google.auth import default
creds, _ = default()
import gspread, openpyxl
try:
    from gspread.utils import ValueRenderOption
    UNF = ValueRenderOption.unformatted
except Exception:
    UNF = 'UNFORMATTED_VALUE'
gc = gspread.authorize(creds)
sh = gc.open_by_key(SHEET_ID)
have = {w.title for w in sh.worksheets()}
wb = openpyxl.Workbook(write_only=True)
t0 = time.time()
for name in TABS:
    if name not in have:
        print('탭 없음(건너뜀):', name); continue
    vals = sh.worksheet(name).get_values(value_render_option=UNF)
    out = wb.create_sheet(title=name)
    for row in vals:
        out.append([None if v == '' else v for v in row])
    print(f'{name}: {len(vals)}행')
wb.save(XLSX)
print(f'사본 저장: {XLSX} ({round(time.time()-t0)}초)')

In [ ]:
#@title 3) 엔진 받기 → 검산 실행
import subprocess, os, shutil
def run(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return r.returncode, (r.stdout or '') + (r.stderr or '')
print(run('node -v')[1].strip(), '/ npm', run('npm -v')[1].strip())
if os.path.isdir(REPO):
    shutil.rmtree(REPO)
rc, out = run(f'git clone -q -b {BRANCH} {REPO_URL} {REPO}')
assert rc == 0, out
rc, out = run('npm install --no-audit --no-fund --silent', cwd=REPO)
assert rc == 0, out
extra = (f' --period "{PERIOD}"' if PERIOD else '') + (f' --cut "{CUT}"' if CUT else '')
rc, out = run(f'node fsg-report/compare-sheet.js --sheet "{XLSX}"{extra}', cwd=REPO)
open(RESULT, 'w', encoding='utf-8').write(out)
print(out)
print('종료 코드', rc, '(0 = 모두 일치, 1 = 차이 있음, 2 = 읽기 오류)')

In [ ]:
#@title 4) 결과 요약을 Drive 폴더 FSG_report_out 에 저장 (이름 없는 요약만)
import datetime
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
drv = build('drive', 'v3', credentials=creds)
q = "name='FSG_report_out' and mimeType='application/vnd.google-apps.folder' and trashed=false"
found = drv.files().list(q=q, fields='files(id)').execute().get('files', [])
fid = found[0]['id'] if found else drv.files().create(body={'name': 'FSG_report_out', 'mimeType': 'application/vnd.google-apps.folder'}, fields='id').execute()['id']
stamp = datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y%m%d_%H%M')
meta = {'name': f'검산결과_{stamp}.txt', 'parents': [fid]}
f = drv.files().create(body=meta, media_body=MediaFileUpload(RESULT, mimetype='text/plain'), fields='id,name,webViewLink').execute()
print('저장:', f['name'], f['webViewLink'])
print('→ Claude에게 "검산 결과 저장했어"라고 알려 주면 Drive에서 읽어 확인합니다.')